# RetailPulse -- Week 1 Checkpoint

**Objective:** Consolidate all Week 1 work into a structured review. Log baseline models and metrics in MLflow for reproducibility. Summarize findings and prepare for Week 2.


In [1]:
import os
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(FIGURES_DIR, exist_ok=True)


In [2]:
def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved: {name}")


## Load Week 1 Outputs

In [3]:
# Load datasets
rfm = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_rfm.csv"))
segments = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_segments.csv"))
daily = pd.read_csv(os.path.join(PROCESSED_DIR, "daily_sales_features.csv"), parse_dates=["Date"])
prophet_fc = pd.read_csv(os.path.join(PROCESSED_DIR, "prophet_forecast_30d.csv"), parse_dates=["ds"])
lstm_pred = pd.read_csv(os.path.join(PROCESSED_DIR, "lstm_predictions.csv"), parse_dates=["ds"])

print("Loaded datasets:")
print(f"  RFM:             {rfm.shape}")
print(f"  Segments:        {segments.shape}")
print(f"  Daily sales:     {daily.shape}")
print(f"  Prophet 30d:     {prophet_fc.shape}")
print(f"  LSTM predictions: {lstm_pred.shape}")


Loaded datasets:
  RFM:             (4312, 10)
  Segments:        (4312, 11)
  Daily sales:     (374, 24)
  Prophet 30d:     (30, 4)
  LSTM predictions: (30, 4)


## MLflow Experiment Logging

Log all models, parameters, and metrics from Week 1 into MLflow for reproducibility and experiment tracking.


In [4]:
# Set up MLflow
mlflow.set_tracking_uri(os.path.join("..", "mlflow", "mlruns"))
experiment_name = "RetailPulse-Week1"
mlflow.set_experiment(experiment_name)
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {experiment_name}")


2026/05/12 10:45:05 INFO mlflow.tracking.fluent: Experiment with name 'RetailPulse-Week1' does not exist. Creating a new experiment.


MLflow tracking URI: ../mlflow/mlruns
Experiment: RetailPulse-Week1


In [5]:
# Log K-Means segmentation
with mlflow.start_run(run_name="kmeans_segmentation"):
    n_clusters = segments["kmeans_cluster"].nunique()
    mlflow.log_param("algorithm", "K-Means")
    mlflow.log_param("n_clusters", n_clusters)
    mlflow.log_param("features", "recency, frequency, monetary")
    mlflow.log_param("scaling", "log1p + StandardScaler")

    segment_counts = segments["kmeans_label"].value_counts()
    for label, count in segment_counts.items():
        mlflow.log_metric(f"segment_{label.replace(' ', '_').replace('/', '_')}_count", count)

    mlflow.log_metric("total_customers", len(segments))

    # Log segment distribution as artifact
    seg_summary = segments.groupby("kmeans_label").agg(
        count=("Customer ID", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
    ).round(2)
    seg_summary.to_csv(os.path.join(FIGURES_DIR, "segment_summary.csv"))
    mlflow.log_artifact(os.path.join(FIGURES_DIR, "segment_summary.csv"))

    print(f"Logged K-Means segmentation: {n_clusters} clusters, {len(segments):,} customers")


Logged K-Means segmentation: 2 clusters, 4,312 customers


In [6]:
# Log Prophet model
with mlflow.start_run(run_name="prophet_baseline"):
    mlflow.log_param("model", "Prophet")
    mlflow.log_param("weekly_seasonality", True)
    mlflow.log_param("monthly_seasonality", True)
    mlflow.log_param("country_holidays", "GB")
    mlflow.log_param("test_days", 30)

    # Compute Prophet metrics from forecast file
    prophet_full = pd.read_csv(os.path.join(PROCESSED_DIR, "prophet_forecast_full.csv"),
                               parse_dates=["ds"])
    prophet_ready = pd.read_csv(os.path.join(PROCESSED_DIR, "prophet_ready.csv"),
                                parse_dates=["ds"])

    # Merge actual with forecast on test period
    test_start = prophet_ready["ds"].max() - pd.Timedelta(days=30)
    actual_test = prophet_ready[prophet_ready["ds"] > test_start]
    pred_test = prophet_full[prophet_full["ds"].isin(actual_test["ds"])]

    if len(pred_test) > 0 and len(actual_test) > 0:
        merged = actual_test.merge(pred_test[["ds", "yhat"]], on="ds")
        actual = merged["y"].values
        predicted = merged["yhat"].values
        non_zero = actual != 0

        if non_zero.sum() > 0:
            mape = np.mean(np.abs((actual[non_zero] - predicted[non_zero]) / actual[non_zero])) * 100
        else:
            mape = 0
        mae = np.mean(np.abs(actual - predicted))
        rmse = np.sqrt(np.mean((actual - predicted) ** 2))

        mlflow.log_metric("MAPE", round(mape, 2))
        mlflow.log_metric("MAE", round(mae, 2))
        mlflow.log_metric("RMSE", round(rmse, 2))
        print(f"Logged Prophet: MAPE={mape:.2f}%, MAE={mae:.2f}, RMSE={rmse:.2f}")
    else:
        print("Could not compute Prophet test metrics, logging forecast summary instead")
        mlflow.log_metric("forecast_mean", round(prophet_fc["yhat"].mean(), 2))

    mlflow.log_artifact(os.path.join(PROCESSED_DIR, "prophet_forecast_30d.csv"))


Logged Prophet: MAPE=21.32%, MAE=9990.18, RMSE=12424.79


In [7]:
# Log LSTM model
with mlflow.start_run(run_name="lstm_forecasting"):
    mlflow.log_param("model", "LSTM")
    mlflow.log_param("input_dim", 7)
    mlflow.log_param("hidden_dim", 64)
    mlflow.log_param("num_layers", 2)
    mlflow.log_param("dropout", 0.2)
    mlflow.log_param("lookback", 30)
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("learning_rate", 0.001)

    actual = lstm_pred["actual"].values
    predicted = lstm_pred["lstm_predicted"].values
    non_zero = actual != 0

    if non_zero.sum() > 0:
        mape = np.mean(np.abs((actual[non_zero] - predicted[non_zero]) / actual[non_zero])) * 100
    else:
        mape = 0
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))

    mlflow.log_metric("MAPE", round(mape, 2))
    mlflow.log_metric("MAE", round(mae, 2))
    mlflow.log_metric("RMSE", round(rmse, 2))

    mlflow.log_artifact(os.path.join(PROCESSED_DIR, "lstm_predictions.csv"))

    print(f"Logged LSTM: MAPE={mape:.2f}%, MAE={mae:.2f}, RMSE={rmse:.2f}")


Logged LSTM: MAPE=20.41%, MAE=11514.43, RMSE=15097.07


## Week 1 Summary Dashboard

In [8]:
fig, axes = plt.subplots(2, 3, figsize=(22, 12))

# 1. Data overview
labels = ["Raw", "Cleaned"]
sizes = [525461, len(daily) * 100]  # approximate
clean_count = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_rfm.csv")).shape[0]
axes[0, 0].bar(["Raw Transactions", "Unique Customers", "Products", "Countries"],
               [525461, clean_count, 4632, 40],
               color=["#3498db", "#2ecc71", "#e74c3c", "#9b59b6"])
axes[0, 0].set_title("Dataset Overview")
axes[0, 0].set_ylabel("Count")
for i, v in enumerate([525461, clean_count, 4632, 40]):
    axes[0, 0].text(i, v + 5000, f"{v:,}", ha="center", fontsize=9)

# 2. Segment distribution
seg_counts = segments["kmeans_label"].value_counts()
colors_seg = plt.cm.Set2(np.linspace(0, 1, len(seg_counts)))
axes[0, 1].pie(seg_counts.values, labels=seg_counts.index, autopct="%1.1f%%",
               colors=colors_seg, textprops={"fontsize": 8})
axes[0, 1].set_title("Customer Segments (K-Means)")

# 3. Revenue trend
axes[0, 2].plot(daily["Date"], daily["total_revenue"], alpha=0.3, color="#bdc3c7", linewidth=0.5)
ma30 = daily["total_revenue"].rolling(30).mean()
axes[0, 2].plot(daily["Date"], ma30, color="#3498db", linewidth=2)
axes[0, 2].set_title("Revenue Trend (30-day MA)")
axes[0, 2].set_ylabel("Revenue")

# 4. RFM score distribution
axes[1, 0].hist(rfm["rfm_score"], bins=13, color="#3498db", edgecolor="white", alpha=0.8)
axes[1, 0].set_title("RFM Score Distribution")
axes[1, 0].set_xlabel("RFM Score")
axes[1, 0].set_ylabel("Customers")

# 5. Prophet forecast
axes[1, 1].plot(prophet_fc["ds"], prophet_fc["yhat"], color="#e74c3c", linewidth=2,
                marker="o", markersize=3)
axes[1, 1].fill_between(prophet_fc["ds"], prophet_fc["yhat_lower"],
                        prophet_fc["yhat_upper"], alpha=0.2, color="#e74c3c")
axes[1, 1].set_title("30-Day Prophet Forecast")
axes[1, 1].set_ylabel("Predicted Revenue")

# 6. LSTM actual vs predicted
axes[1, 2].plot(lstm_pred["ds"], lstm_pred["actual"], color="#2c3e50",
                linewidth=1.5, label="Actual")
axes[1, 2].plot(lstm_pred["ds"], lstm_pred["lstm_predicted"], color="#e74c3c",
                linewidth=1.5, label="LSTM")
axes[1, 2].set_title("LSTM Test Performance")
axes[1, 2].set_ylabel("Revenue")
axes[1, 2].legend(fontsize=9)

fig.suptitle("RetailPulse -- Week 1 Checkpoint Summary",
             fontsize=18, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "30_week1_summary.png")
plt.show()


Saved: 30_week1_summary.png


In [9]:
# Consolidated metrics table
actual_lstm = lstm_pred["actual"].values
pred_lstm = lstm_pred["lstm_predicted"].values
nz = actual_lstm != 0
lstm_mape = np.mean(np.abs((actual_lstm[nz] - pred_lstm[nz]) / actual_lstm[nz])) * 100
lstm_mae = np.mean(np.abs(actual_lstm - pred_lstm))
lstm_rmse = np.sqrt(np.mean((actual_lstm - pred_lstm) ** 2))

metrics_summary = {
    "Component": ["Customer Segmentation", "Customer Segmentation",
                   "Prophet Forecasting", "Prophet Forecasting", "Prophet Forecasting",
                   "LSTM Forecasting", "LSTM Forecasting", "LSTM Forecasting"],
    "Metric": ["K-Means Clusters", "Total Customers",
               "MAPE (%)", "MAE", "RMSE",
               "MAPE (%)", "MAE", "RMSE"],
    "Value": [segments["kmeans_cluster"].nunique(), len(segments),
              round(mape, 2), round(mae, 2), round(rmse, 2),
              round(lstm_mape, 2), round(lstm_mae, 2), round(lstm_rmse, 2)],
}
metrics_df = pd.DataFrame(metrics_summary)
print("WEEK 1 -- CONSOLIDATED METRICS")
print("=" * 55)
print(metrics_df.to_string(index=False))


WEEK 1 -- CONSOLIDATED METRICS
            Component           Metric    Value
Customer Segmentation K-Means Clusters     2.00
Customer Segmentation  Total Customers  4312.00
  Prophet Forecasting         MAPE (%)    20.41
  Prophet Forecasting              MAE 11514.43
  Prophet Forecasting             RMSE 15097.07
     LSTM Forecasting         MAPE (%)    20.41
     LSTM Forecasting              MAE 11514.43
     LSTM Forecasting             RMSE 15097.07


## Week 1 Deliverables Checklist

In [10]:
deliverables = [
    ("EDA notebook with 7 visualizations", True),
    ("Data cleaning pipeline (525K -> ~390K rows)", True),
    ("RFM feature engineering with scoring", True),
    ("Customer segmentation (K-Means + DBSCAN)", True),
    ("Time-series preparation with stationarity tests", True),
    ("Seasonal decomposition (weekly + monthly)", True),
    ("Baseline Prophet model with parameter tuning", True),
    ("LSTM model with PyTorch", True),
    ("30-day future demand forecast", True),
    ("MLflow experiment logging", True),
    ("All figures saved to reports/figures/", True),
    ("Processed datasets exported", True),
]

print("WEEK 1 DELIVERABLES")
print("=" * 55)
for item, done in deliverables:
    status = "DONE" if done else "PENDING"
    print(f"  [{status}] {item}")
print()
print(f"Total figures generated: {len([f for f in os.listdir(FIGURES_DIR) if f.endswith('.png')])} PNG files")
print(f"Total processed files: {len([f for f in os.listdir(PROCESSED_DIR) if f.endswith('.csv')])} CSV files")
print()
print("Week 2 plan:")
print("  - Hybrid Prophet + LSTM ensemble forecasting")
print("  - Churn prediction model with XGBoost + SHAP")
print("  - Inventory optimization logic")
print("  - Feature importance analysis with Optuna tuning")
print("  - Drift detection with Evidently AI")


WEEK 1 DELIVERABLES
  [DONE] EDA notebook with 7 visualizations
  [DONE] Data cleaning pipeline (525K -> ~390K rows)
  [DONE] RFM feature engineering with scoring
  [DONE] Customer segmentation (K-Means + DBSCAN)
  [DONE] Time-series preparation with stationarity tests
  [DONE] Seasonal decomposition (weekly + monthly)
  [DONE] Baseline Prophet model with parameter tuning
  [DONE] LSTM model with PyTorch
  [DONE] 30-day future demand forecast
  [DONE] MLflow experiment logging
  [DONE] All figures saved to reports/figures/
  [DONE] Processed datasets exported

Total figures generated: 30 PNG files
Total processed files: 9 CSV files

Week 2 plan:
  - Hybrid Prophet + LSTM ensemble forecasting
  - Churn prediction model with XGBoost + SHAP
  - Inventory optimization logic
  - Feature importance analysis with Optuna tuning
  - Drift detection with Evidently AI
